# 02_rand_inspection

RAND: classificação CCW, antecedentes e auditoria. Notebook exploratório reproduzível (dados `data/raw` e `data/processed` não estão versionados). Execute com diretório de trabalho `notebooks/`. Resultados são preditivos/descritivos, **não causais**.

In [ ]:
from pathlib import Path
import pandas as pd

arquivo = Path(
    "../reports/attachments/"
    "TL-A243-2-v4 State Firearm Law Database 6.0.xlsx"
)

excel = pd.ExcelFile(arquivo)

print("Abas encontradas:")
for aba in excel.sheet_names:
    print("-", aba)

In [ ]:
# Carregar a aba completa
leis = pd.read_excel(
    arquivo,
    sheet_name="Database"
)

print("Total de registros:", len(leis))
print("Estados encontrados:", leis["State"].nunique())

print("\nCategorias de legislação:")
display(
    leis["Law Class"]
    .value_counts()
    .to_frame("Quantidade")
)

In [ ]:
porte = leis[
    leis["Law Class"] == "carrying a concealed weapon (ccw)"
].copy()

colunas = [
    "State",
    "Law Class Subtype",
    "Type of Change",
    "Effective Date Year",
    "Supersession Date Year"
]

display(
    porte[porte["State"] == "Alaska"]
    [colunas]
    .sort_values("Effective Date Year")
)

In [ ]:
colunas = [
    "Law ID",
    "State",
    "Law Class Subtype",
    "Effect",
    "Type of Change",
    "Effective Date Year",
    "Supersession Date Year"
]

louisiana = porte[
    porte["State"] == "Louisiana"
].copy()

display(
    louisiana[colunas]
    .sort_values("Effective Date Year")
)

In [ ]:
registros = []

for estado in porte["State"].unique():
    dados_estado = porte[porte["State"] == estado]

    for ano in range(2019, 2025):
        vigentes = dados_estado[
            (dados_estado["Effective Date Year"] <= ano)
            & (
                dados_estado["Supersession Date Year"].isna()
                | (dados_estado["Supersession Date Year"] > ano)
            )
        ]

        modalidades = set(vigentes["Law Class Subtype"])

        registros.append({
            "state": estado,
            "year": ano,
            "ccw_shall_issue": int("shall issue" in modalidades),
            "ccw_permitless": int(
                "shall issue (permit not required)" in modalidades
            ),
            "ccw_records": len(vigentes)
        })

df_ccw = pd.DataFrame(registros)

print("Total de registros:", len(df_ccw))
print("Estados encontrados:", df_ccw["state"].nunique())

display(df_ccw.head(10))

In [ ]:
print("Registros sem legislação CCW identificada:")
display(df_ccw[df_ccw["ccw_records"] == 0])

print("\nRegistros por ano:")
display(df_ccw.groupby("year")["state"].nunique())

print("\nDuplicações:")
print(df_ccw.duplicated(["state", "year"]).sum())

In [ ]:
for modalidade, coluna in [
    ("may issue", "ccw_may_issue"),
    ("prohibited", "ccw_prohibited")
]:
    registros_coluna = []

    for _, linha in df_ccw.iterrows():
        vigentes = porte[
            (porte["State"] == linha["state"])
            & (porte["Effective Date Year"] <= linha["year"])
            & (
                porte["Supersession Date Year"].isna()
                | (porte["Supersession Date Year"] > linha["year"])
            )
        ]

        registros_coluna.append(
            int(modalidade in set(vigentes["Law Class Subtype"]))
        )

    df_ccw[coluna] = registros_coluna

# Remover Washington, D.C. para manter os 50 estados
df_ccw = df_ccw[df_ccw["state"] != "District of Columbia"].copy()

print("Dimensões:", df_ccw.shape)

display(df_ccw.head(10))

In [ ]:
colunas_ccw = [
    "ccw_shall_issue",
    "ccw_permitless",
    "ccw_may_issue",
    "ccw_prohibited"
]

print("Estados:", df_ccw["state"].nunique())
print("Registros:", len(df_ccw))

print("\nRegistros sem nenhuma modalidade:")
print((df_ccw[colunas_ccw].sum(axis=1) == 0).sum())

print("\nRegistros com múltiplas modalidades:")
print((df_ccw[colunas_ccw].sum(axis=1) > 1).sum())

In [ ]:
from pathlib import Path

destino = Path("../data/processed/rand_ccw_2019_2024.csv")

colunas_exportacao = [
    "state",
    "year",
    "ccw_shall_issue",
    "ccw_permitless",
    "ccw_may_issue",
    "ccw_prohibited"
]

df_ccw[colunas_exportacao].to_csv(destino, index=False)

print("Arquivo salvo em:", destino.resolve())

In [ ]:
background = leis[
    leis["Law Class"] == "background checks"
].copy()

print("Registros:", len(background))

display(
    background[
        [
            "Law Class Subtype",
            "Type of Change",
            "Effective Date Year",
            "Supersession Date Year"
        ]
    ].head(10)
)

In [ ]:
vendas_privadas = background[
    background["Law Class Subtype"] == "private sales : point of sale"
].copy()

print("Registros:", len(vendas_privadas))
print("Estados:", vendas_privadas["State"].nunique())

print("\nTipos de alteração:")
display(vendas_privadas["Type of Change"].value_counts(dropna=False))

print("\nExemplo de registros:")
display(
    vendas_privadas[
        [
            "State",
            "Effect",
            "Type of Change",
            "Effective Date Year",
            "Supersession Date Year"
        ]
    ].head(10)
)

In [ ]:
revogacoes = vendas_privadas[
    vendas_privadas["Type of Change"] == "Repeal"
].copy()

colunas = [
    "Law ID",
    "State",
    "Handguns or Long Guns",
    "Effect",
    "Type of Change",
    "Effective Date Year",
    "Supersession Date Year"
]

display(
    revogacoes[colunas]
    .sort_values(["State", "Effective Date Year"])
)

In [ ]:
nc = vendas_privadas[
    vendas_privadas["State"] == "North Carolina"
].copy()

colunas = [
    "Law ID",
    "Handguns or Long Guns",
    "Effect",
    "Type of Change",
    "Effective Date Year",
    "Supersession Date Year",
]

display(
    nc[colunas]
    .sort_values("Effective Date Year")
)

In [ ]:
registros_background = []

# Usar os 50 estados já identificados no dataset CCW
for estado in sorted(df_ccw["state"].unique()):

    dados_estado = vendas_privadas[
        vendas_privadas["State"] == estado
    ]

    for ano in range(2019, 2025):

        vigentes = dados_estado[
            (dados_estado["Effective Date Year"] <= ano)
            & (
                dados_estado["Supersession Date Year"].isna()
                | (dados_estado["Supersession Date Year"] > ano)
            )
            & (dados_estado["Type of Change"] != "Repeal")
        ]

        tipos = set(vigentes["Handguns or Long Guns"])

        registros_background.append({
            "state": estado,
            "year": ano,
            "background_handgun": int(
                bool(tipos & {"handgun", "handgun and long gun"})
            ),
            "background_longgun": int(
                bool(tipos & {"long gun", "handgun and long gun"})
            )
        })

df_background = pd.DataFrame(registros_background)

print("Dimensões:", df_background.shape)

display(df_background.head(10))

In [ ]:
print("Estados:", df_background["state"].nunique())
print("Registros:", len(df_background))

print("\nDuplicações:")
print(df_background.duplicated(["state", "year"]).sum())

print("\nNorth Carolina:")
display(
    df_background[
        df_background["state"] == "North Carolina"
    ]
)

print("\nEstados com exigências identificadas em 2024:")
display(
    df_background[df_background["year"] == 2024][
        ["background_handgun", "background_longgun"]
    ].sum()
)

In [ ]:
from pathlib import Path

destino = Path("../data/processed/rand_background_2019_2024.csv")

df_background.to_csv(destino, index=False)

print("Arquivo salvo em:", destino.resolve())

In [ ]:
# Nome correto da categoria RAND, ao invés de procurar a palavra 'permitless'
estados_permitless = ["Arkansas","Iowa","Montana","Tennessee","Texas","Utah","Georgia","Indiana","Ohio","Alabama","Florida","Nebraska"]
registros_permitless = leis[leis["State"].isin(estados_permitless) & leis["Law Class Subtype"].eq("shall issue (permit not required)") & leis["Effective Date Year"].between(2020, 2023)]
display(registros_permitless[["Law ID","State","Law Class Subtype","Type of Change","Effective Date Year","Effective Date Month","Effective Date Day"]].sort_values(["Effective Date Year","State"]))
assert registros_permitless["State"].nunique() == 12
